# gold_revenue_per_restaurant

**Sources:**
- fact:      `silver.order_items`   (order_item_id, order_id, subtotal, discount_applied)
- bridge:    `silver.orders`        (order_id → restaurant_key which is the CNPJ)
- dimension: `silver.restaurants`   (cnpj → name, city, cuisine_type)

**Joins:**
- `order_items.order_id = orders.order_id`
- `orders.restaurant_key = restaurants.cnpj`  (CNPJ is the business key — CLAUDE.md)

**Grain:** one row per restaurant CNPJ  
**Merge key:** `restaurant_cnpj`  
**Lineage:** Silver → Gold (never Bronze directly)

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "ubereats_dev")

In [ ]:
catalog              = dbutils.widgets.get("catalog")
gold_table           = f"{catalog}.gold.revenue_per_restaurant"
silver_order_items   = f"{catalog}.silver.order_items"
silver_orders        = f"{catalog}.silver.orders"
silver_restaurants   = f"{catalog}.silver.restaurants"

print(f"[gold] {gold_table}  ←  {silver_order_items} × {silver_orders} × {silver_restaurants}")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_table} (
        restaurant_cnpj    STRING NOT NULL,
        restaurant_name    STRING,
        city               STRING,
        cuisine_type       STRING,
        total_orders       LONG,
        total_items_sold   LONG,
        total_revenue_brl  DOUBLE,
        avg_order_value_brl DOUBLE,
        avg_discount_brl   DOUBLE,
        _computed_at       TIMESTAMP NOT NULL
    ) USING DELTA
    CLUSTER BY (restaurant_cnpj)
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"[gold] table {gold_table} ready")

In [ ]:
from pyspark.sql.functions import col, count, sum, avg, countDistinct, current_timestamp

order_items  = spark.table(silver_order_items)
orders       = spark.table(silver_orders)
restaurants  = spark.table(silver_restaurants)

# Join chain:
#   fact silver.order_items
#     ← order_id →  bridge silver.orders  (carries restaurant_key = CNPJ)
#     ← restaurant_key = cnpj →  dimension silver.restaurants
revenue_df = (
    order_items.alias("i")
    .join(
        orders.select(
            col("order_id"),
            col("restaurant_key"),
            col("total_amount"),
        ).alias("o"),
        col("i.order_id") == col("o.order_id"),
        "inner",
    )
    .join(
        restaurants.select(
            col("cnpj"),
            col("name").alias("restaurant_name"),
            col("city"),
            col("cuisine_type"),
        ).alias("r"),
        col("o.restaurant_key") == col("r.cnpj"),
        "left",
    )
    .groupBy(
        col("o.restaurant_key").alias("restaurant_cnpj"),
        col("r.restaurant_name"),
        col("r.city"),
        col("r.cuisine_type"),
    )
    .agg(
        countDistinct(col("i.order_id")).alias("total_orders"),
        count(col("i.order_item_id")).alias("total_items_sold"),
        sum(col("i.subtotal")).alias("total_revenue_brl"),
        avg(col("o.total_amount")).alias("avg_order_value_brl"),
        avg(col("i.discount_applied")).alias("avg_discount_brl"),
    )
    .withColumn("_computed_at", current_timestamp())
)

print(f"[gold] {revenue_df.count()} restaurants")

In [ ]:
revenue_df.createOrReplaceTempView("gold_revenue_per_restaurant_batch")

spark.sql(f"""
    MERGE INTO {gold_table} AS t
    USING gold_revenue_per_restaurant_batch AS s
    ON t.restaurant_cnpj = s.restaurant_cnpj
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"[gold] {gold_table} updated")